# MMaDA-space
Official MMaDA understanding, image generation, and hidden-state capture. Use a fresh **A100 80 GB** runtime with BF16, without quantization. Publish these files to `main` before opening this notebook from GitHub.

## 1. Setup
Run once in a fresh runtime. Restart the session after installing dependencies. PyTorch and torchvision come from Colab; no FlashAttention installation is needed.

In [ ]:
from pathlib import Path

PROJECT = Path("/content/MMaDa-space")
UPSTREAM = Path("/content/MMaDA")
!git clone --branch main https://github.com/Nicerova7/MMaDa-space.git /content/MMaDa-space
!git clone https://github.com/Gen-Verse/MMaDA.git /content/MMaDA
!git -C /content/MMaDA checkout 3cdb8709635beb29f66e4e23cd66060872ac3996
%pip install -r /content/MMaDa-space/requirements.txt


## 2. Load the model
Run after restarting. This adapter is reused for U and G.

In [ ]:
import os
import sys
from pathlib import Path

import torch
from IPython.display import display

PROJECT = Path("/content/MMaDa-space")
UPSTREAM = Path("/content/MMaDA")
os.chdir(PROJECT)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = str(PROJECT) + os.pathsep + str(UPSTREAM)
sys.path[:0] = [str(PROJECT), str(UPSTREAM)]

from mmada_space.adapter import MMaDAAdapter
from mmada_space.capture import Capture

adapter = MMaDAAdapter()
print(torch.cuda.get_device_name(0), adapter.dtype)
print("Layers:", len(adapter.layers), "| Hidden size:", adapter.model.config.d_model)


## 3. Understanding
### 3.1. Upload one image
The official preprocessing resizes the shortest side and center-crops to 512 ? 512.

In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()
image_name = next(iter(uploaded))
u_image = PROJECT / "inputs/images" / Path(image_name).name
u_image.write_bytes(uploaded[image_name])
with Image.open(u_image) as image:
    display(image.copy())


### 3.2. Prepare the input and choose capture settings
Layer indices start at 0; capture steps start at 1. U also uses denoising. `max_new_tokens` must be divisible by `block_length`; `steps` must be divisible by the resulting number of response blocks.

In [ ]:
u_question = "Describe this image."
u_layers = [0, 15, 31]  # list(range(len(adapter.layers))) for all layers
u_capture_step = 1
u_settings = dict(max_new_tokens=128, steps=64, block_length=32, seed=42)
U_DIR = PROJECT / "outputs/understanding"

u_input_ids = adapter.prepare_understanding(u_image, u_question)
print("Input tokens:", u_input_ids.shape[1])


### 3.3. Answer and capture
Captures layer outputs during the selected U evaluation, including visual tokens, prompt tokens and response positions. Response positions may still contain mask tokens.

In [ ]:
with Capture(adapter, u_layers, step=u_capture_step) as u_capture:
    answer = adapter.answer(u_input_ids, **u_settings)

print(f"U | capture step {u_capture_step}/{u_capture.evaluations} | layers {u_layers}")
print(answer)


### 3.4. Save the answer, states and metadata


In [ ]:
u_metadata = dict(
    **adapter.metadata(), task="understanding", image=str(u_image),
    question=u_question, answer=answer, **u_settings,
    temperature=0.0, cfg_scale=0.0, remasking="low_confidence",
    alignment=adapter.alignment(u_capture.input_ids, u_input_ids.shape[1]),
    preprocessing="RGB; bicubic shortest-side resize; 512 center crop; [-1, 1]",
)
u_capture.save(U_DIR, u_metadata)
(U_DIR / "answer.txt").write_text(answer + "\n", encoding="utf-8")
print("Saved:", U_DIR)


## 4. Generation
### 4.1. Choose the prompt, layers and capture step
Defaults follow the official README example: 15 steps and guidance 3.5. Its guidance formula is `(1 + guidance) * conditional - guidance * unconditional`.

In [ ]:
g_prompt = "A curly-haired blonde girl smiles softly, her bright eyes joyful. She wears a pink lace dress, with rosy cheeks against a flowered background."
g_layers = [0, 15, 31]
g_capture_step = 1
g_settings = dict(steps=15, guidance=3.5, seed=42)
G_DIR = PROJECT / "outputs/generation"


### 4.2. Generate and capture
Captures conditional CFG row 0 only, before the selected sampling update. The states contain prompt and visual positions. No extra forward pass is added.

In [ ]:
with Capture(adapter, g_layers, step=g_capture_step) as g_capture:
    image = adapter.generate(g_prompt, **g_settings)

print(f"G | capture step {g_capture_step}/{g_capture.evaluations} | layers {g_layers}")
display(image)


### 4.3. Save the image, states and metadata


In [ ]:
g_metadata = dict(
    **adapter.metadata(), task="generation", prompt=g_prompt, **g_settings,
    temperature=1.0, mask_schedule="cosine",
    cfg_formula="(1 + guidance) * conditional - guidance * unconditional",
    alignment=adapter.alignment(g_capture.input_ids, len(g_capture.input_ids)),
)
g_capture.save(G_DIR, g_metadata)
image.save(G_DIR / "image.png")
print("Saved:", G_DIR)


## 5. Inspect saved states
CPU float32 tensors have shape `[tokens, hidden]`. No token pooling is applied; `input_ids` records the exact input at the captured evaluation.

In [ ]:
u_saved = torch.load(U_DIR / "hidden_states.pt", map_location="cpu", weights_only=True)
g_saved = torch.load(G_DIR / "hidden_states.pt", map_location="cpu", weights_only=True)
for task, saved in (("U", u_saved), ("G", g_saved)):
    print(task, "| step", saved["metadata"]["capture_step"], "| evaluations", saved["metadata"]["evaluations"])
    print("Alignment:", saved["metadata"]["alignment"])
    for layer, states in saved["hidden_states"].items():
        print("Layer:", layer, "| Shape:", tuple(states.shape))


## 6. Download results
Includes the input image, answer, generated image, states, and metadata. Download before disconnecting the runtime.

In [ ]:
import shutil

archive_root = PROJECT / "outputs/base-run"
archive_root.mkdir(exist_ok=True)
shutil.copytree(U_DIR, archive_root / "understanding", dirs_exist_ok=True)
shutil.copytree(G_DIR, archive_root / "generation", dirs_exist_ok=True)
shutil.copy2(u_image, archive_root / ("input" + u_image.suffix))
archive = shutil.make_archive("/content/mmada-base-run", "zip", archive_root)
files.download(archive)
